[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HisameOgasahara/deep-learning-diagnostics-and-improvement/blob/main/practice/01_tensor_memory_layout_and_ops.ipynb)

# 01. Tensor memory layout and core ops

뒤의 모든 노트북에서 반복해서 등장하는 tensor shape, stride, view, transpose, broadcast, reduction, gather/scatter, matmul의 실제 동작을 작은 숫자로 확인한다.

**반복 형식:** 바닐라 PyTorch 실행 → profiler로 ATen/CUDA 연산 확인 → 필요할 때만 작은 텐서로 수학적 전개를 펼친다.


In [1]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(7)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)
print("torch:", torch.__version__)


device: cuda
torch: 2.11.0+cu128


In [2]:
from torch.profiler import profile, ProfilerActivity

def profile_call(name, fn, *args, **kwargs):
    activities = [ProfilerActivity.CPU]
    if torch.cuda.is_available():
        activities.append(ProfilerActivity.CUDA)

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    with profile(
        activities=activities,
        record_shapes=True,
        profile_memory=True,
        with_stack=False,
    ) as prof:
        out = fn(*args, **kwargs)

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    print(f"\n[{name}] top operators")
    sort_key = "self_cuda_time_total" if torch.cuda.is_available() else "self_cpu_time_total"
    print(prof.key_averages().table(sort_by=sort_key, row_limit=12))

    return out


## 1. Shape, stride, storage


In [3]:
x = torch.arange(12, device=device).reshape(3, 4)
print(x)
print("shape:", x.shape)
print("stride:", x.stride())
print("contiguous:", x.is_contiguous())


tensor([[ 0,  1,  2,  3],
        [ 4,  5,  6,  7],
        [ 8,  9, 10, 11]], device='cuda:0')
shape: torch.Size([3, 4])
stride: (4, 1)
contiguous: True


## 2. View / reshape / transpose / contiguous


In [4]:
xt = x.transpose(0, 1)
print("transposed stride:", xt.stride())
print("transposed contiguous:", xt.is_contiguous())

xc = xt.contiguous()
print("after contiguous stride:", xc.stride())
print("after contiguous:", xc.is_contiguous())

flat = xc.view(-1)
print("flat:", flat)


transposed stride: (1, 4)
transposed contiguous: False
after contiguous stride: (3, 1)
after contiguous: True
flat: tensor([ 0,  4,  8,  1,  5,  9,  2,  6, 10,  3,  7, 11], device='cuda:0')


In [5]:
_ = profile_call("transpose only", lambda z: z.transpose(0, 1), x)
_ = profile_call("transpose + contiguous", lambda z: z.transpose(0, 1).contiguous(), x)



[transpose only] top operators
---------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                       Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg       CPU Mem  Self CPU Mem    # of Calls  
---------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
            aten::transpose         1.23%      32.871us         1.49%      39.928us      39.928us           0 B           0 B             1  
           aten::as_strided         0.26%       7.057us         0.26%       7.057us       7.057us           0 B           0 B             1  
      cudaDeviceSynchronize        22.81%     610.208us        22.81%     610.208us     610.208us           0 B           0 B             1  
    Activity Buffer Request        75.70%       2.025ms        75.70%       2.025ms       2.025ms           0 B     

/usr/local/lib/python3.13/dist-packages/torch/profiler/profiler.py:224: UserWarning: Warning: Profiler clears events at the end of each cycle.Only events from the current cycle will be reported.To keep events across cycles, set acc_events=True.
  _warn_once(


## 3. Broadcasting and reduction


In [6]:
a = torch.tensor([[1.0], [2.0], [3.0]], device=device)
b = torch.tensor([[10.0, 20.0, 30.0, 40.0]], device=device)

y = a + b
print(y)
print("row mean:", y.mean(dim=1))
print("column sum:", y.sum(dim=0))


tensor([[11., 21., 31., 41.],
        [12., 22., 32., 42.],
        [13., 23., 33., 43.]], device='cuda:0')
row mean: tensor([26., 27., 28.], device='cuda:0')
column sum: tensor([ 36.,  66.,  96., 126.], device='cuda:0')


In [7]:
_ = profile_call("broadcast add + reduction", lambda p, q: (p + q).mean(dim=1), a, b)



[broadcast add + reduction] top operators
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg       CPU Mem  Self CPU Mem      CUDA Mem  Self CUDA Mem    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                             aten::mean         1.87%      42.437us         2.77%      62.814us      62.814us       8.576us        69.79%       8.576us       8.576us           0

## 4. Indexing, gather, scatter


In [8]:
src = torch.tensor([[10., 11., 12.], [20., 21., 22.]], device=device)
idx = torch.tensor([[2, 0], [1, 2]], device=device)

g = torch.gather(src, dim=1, index=idx)
print("gather result:\n", g)

base = torch.zeros(2, 3, device=device)
s = base.scatter(1, idx, g)
print("scatter result:\n", s)


gather result:
 tensor([[12., 10.],
        [21., 22.]], device='cuda:0')
scatter result:
 tensor([[10.,  0., 12.],
        [ 0., 21., 22.]], device='cuda:0')


In [9]:
_ = profile_call("gather", lambda z, i: torch.gather(z, 1, i), src, idx)
_ = profile_call("scatter", lambda z, i, v: z.scatter(1, i, v), base, idx, g)



[gather] top operators
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg       CPU Mem  Self CPU Mem      CUDA Mem  Self CUDA Mem    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                           aten::gather         3.35%     134.080us        99.79%       3.993ms       3.993ms       4.320us       100.00%       8.640us       8.640us           0 B           0 B   

## 5. matmul / bmm / einsum


In [10]:
A = torch.arange(12, dtype=torch.float32, device=device).reshape(3, 4)
B = torch.arange(8, dtype=torch.float32, device=device).reshape(4, 2)

print("matmul:\n", A @ B)

Ab = A.unsqueeze(0).repeat(2, 1, 1)
Bb = B.unsqueeze(0).repeat(2, 1, 1)
print("bmm shape:", torch.bmm(Ab, Bb).shape)

print("einsum equals matmul:",
      torch.allclose(torch.einsum("ik,kj->ij", A, B), A @ B))


matmul:
 tensor([[ 28.,  34.],
        [ 76.,  98.],
        [124., 162.]], device='cuda:0')
bmm shape: torch.Size([2, 3, 2])
einsum equals matmul: True


In [11]:
_ = profile_call("matmul", lambda p, q: p @ q, A, B)
_ = profile_call("bmm", lambda p, q: torch.bmm(p, q), Ab, Bb)
_ = profile_call("einsum", lambda p, q: torch.einsum("ik,kj->ij", p, q), A, B)



[matmul] top operators
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg       CPU Mem  Self CPU Mem      CUDA Mem  Self CUDA Mem    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                               aten::mm         4.42%     206.662us        99.10%       4.633ms       4.633ms       5.536us       100.00%      11.072us      11.072us           0 B           0 B   

## References and provenance

**[1.1] Tensor views and strides**
- 출처: PyTorch tensor semantics / ATen
- 이 노트북에서 가져온 부분: shape·stride와 view의 관계

**[1.2] Gather / scatter**
- 출처: PyTorch ATen indexing operators
- 이 노트북에서 가져온 부분: MoE와 3D scatter에서 반복되는 핵심 연산
